In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Training Walkthrough — BrugadaCNN Ensemble\n",
    "**IDSC 2026 | Team MediScript**\n",
    "\n",
    "This notebook documents the training pipeline. For full training runs, use the scripts in `scripts/`.\n",
    "\n",
    "## Pipeline Summary\n",
    "1. Load stratified splits from `outputs/splits.json`\n",
    "2. Train 3 BrugadaCNN models (seeds 42, 123, 7)\n",
    "3. Save best checkpoint per seed by validation AUROC\n",
    "4. Average predictions for ensemble inference\n",
    "\n",
    "## Key Design Decisions\n",
    "| Decision | Choice | Reason |\n",
    "|----------|--------|---------|\n",
    "| Loss function | Weighted BCE (w⁺=3.77) | 3.37:1 class imbalance |\n",
    "| Optimizer | AdamW + Cosine Annealing | Better generalization on small datasets |\n",
    "| Early stopping | Val AUROC, patience=15 | Prevents overfitting |\n",
    "| Batch size | 16 | ~16 steps/epoch with 253 training samples |\n",
    "| Ensemble | 3 seeds averaged | Reduces initialization variance (±0.091 AUROC) |\n",
    "\n",
    "## Final Results\n",
    "| Seed | Val AUROC | Test AUROC |\n",
    "|------|-----------|------------|\n",
    "| 42   | 0.9112    | 0.9264     |\n",
    "| 123  | 0.9649    | 0.9535     |\n",
    "| 7    | 0.8988    | 0.9729     |\n",
    "| **Ensemble** | **0.9483** | **0.9612** |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys, os\n",
    "sys.path.insert(0, os.path.dirname(os.getcwd()))\n",
    "\n",
    "import json\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "\n",
    "plt.rcParams['figure.dpi'] = 120\n",
    "\n",
    "# Load training history\n",
    "with open('../outputs/training_history.json') as f:\n",
    "    history = json.load(f)\n",
    "\n",
    "epochs = range(1, len(history['train_loss']) + 1)\n",
    "fig, axes = plt.subplots(1, 2, figsize=(13, 4))\n",
    "\n",
    "axes[0].plot(epochs, history['train_loss'], label='Train', color='steelblue')\n",
    "axes[0].plot(epochs, history['val_loss'],   label='Val',   color='crimson')\n",
    "axes[0].set_title('Loss Curve', fontweight='bold')\n",
    "axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')\n",
    "axes[0].legend(); axes[0].grid(alpha=0.3)\n",
    "\n",
    "axes[1].plot(epochs, history['train_auroc'], label='Train', color='steelblue')\n",
    "axes[1].plot(epochs, history['val_auroc'],   label='Val',   color='crimson')\n",
    "best_val = max(history['val_auroc'])\n",
    "axes[1].axhline(y=best_val, color='green', linestyle='--', alpha=0.7,\n",
    "                label=f'Best Val: {best_val:.4f}')\n",
    "axes[1].set_title('AUROC Curve', fontweight='bold')\n",
    "axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUROC')\n",
    "axes[1].legend(); axes[1].grid(alpha=0.3)\n",
    "\n",
    "plt.suptitle('BrugadaCNN Training History (Last Run)', fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "print(f'Best validation AUROC: {best_val:.4f}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Run Full Ensemble Training\n",
    "To retrain from scratch, run from the repo root:\n",
    "```bash\n",
    "python scripts/run_ensemble_training.py\n",
    "```\n",
    "Expected runtime: ~20 minutes on CPU, ~5 minutes on GPU."
   ]
  }
 ],
 "metadata": {
  "kernelspec": { "display_name": "Python 3", "language": "python", "name": "python3" },
  "language_info": { "name": "python", "version": "3.12.0" }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}